<a href="https://colab.research.google.com/github/pcmay/ALyzer3D.AI/blob/main/ALyzer3DAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



<div style="display: flex; justify-content: space-between; align-items: center;">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ALyzer3D.AI_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ColabFold_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
</div>



Welcome to **ALyzer3D.AI**. This notebook allows you to predict the amyloidogenicity of a VL domain sequence of a light chain by first generating its 3D structure with [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb) and then automatically analyzing it with the ALyzer3D.AI model.

**Instructions:**

1. **Enter Your Sequence**: In the first cell (query_sequence), paste the amino acid sequence of your light chain's VL domain.
2. **Select a GPU**: Click Runtime, select Change runtime type, select T4 GPU (or any GPU option available). Click Save.
3. **Run Everything**: Click on the menu Runtime -> Run all.

The notebook will now execute all the steps for you: it will install dependencies, run the ColabFold structure prediction (ca. 5 min), and finally, perform the ALyzer3D.AI analysis on the resulting top-ranked structure. The final prediction will be displayed at the bottom of the page.



In [ ]:
#@title Run ColabFold v1.5.5 Prediction

#@markdown ### Enter your protein sequence
query_sequence = 'DIRLTQSPSSLSASVGDRVTITCQASQHINNYLNWYQHKPGQAPKVLIYDASNLATGVPSRFSGNGSGTHFTLTINSLQPEDAATYYCQQHDDLPLTFGGGTKVEIR' #@param {type:"string"}


# --- Standard Parameters (do not change unless you know what you are doing) ---
jobname = 'colabfold_prediction'
num_relax = 0
template_mode = "none"
msa_mode = "mmseqs2_uniref_env"
model_type = "auto"
pair_mode = "unpaired_paired"
num_recycles = 3
# -----------------------------------------------------------------------------

import os
import sys
from sys import version_info

# Check if ColabFold and its dependencies are already installed
if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    # Install ColabFold
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")

    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # !! THIS IS THE FIX for the TensorFlow "undefined symbol" error !!
    # !! We remove the problematic library file that causes the crash. !!
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")

    # Create symbolic links for easier access
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
    # Mark installation as complete
    os.system("touch COLABFOLD_READY")

# --- Code Execution Starts Here ---
import re
import hashlib
from pathlib import Path
import warnings
from Bio import BiopythonDeprecationWarning

# Suppress unnecessary warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)

from colabfold.download import download_alphafold_params
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

# Define a function to create a unique jobname
def get_unique_jobname(basename, sequence):
    # Create a hash from the sequence to make the jobname unique
    sequence_hash = hashlib.sha1(sequence.encode()).hexdigest()[:5]
    unique_name = f"{basename}_{sequence_hash}"

    # Handle potential jobname collisions
    if os.path.exists(unique_name):
        n = 0
        while os.path.exists(f"{unique_name}_{n}"):
            n += 1
        unique_name = f"{unique_name}_{n}"
    return unique_name

# Sanitize and prepare inputs
query_sequence = "".join(query_sequence.split())
sanitized_jobname = re.sub(r'\W+', '', jobname)
jobname = get_unique_jobname(sanitized_jobname, query_sequence)

# Create directory for results
os.makedirs(jobname, exist_ok=True)

# Write sequence to a CSV file for ColabFold
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
    text_file.write(f"id,sequence\n{jobname},{query_sequence}")

print(f"Starting prediction for job: {jobname}")
print(f"Sequence length: {len(query_sequence.replace(':', ''))}")

# Set up logging
result_dir = Path(jobname)
setup_logging(result_dir.joinpath("log.txt"))

# Parse inputs for the ColabFold `run` function
queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)
num_recycles_parsed = None if num_recycles == "auto" else int(num_recycles)
use_templates = template_mode != "none"

# Download AlphaFold parameters
print("Downloading model parameters...")
download_alphafold_params(model_type, Path("."))

# Execute the main prediction function
print("Running prediction...")
run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=None,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,  # Standard number of models
    num_recycles=3,
    num_seeds=1,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    stop_at_score=100.0,
    zip_results=False, # We will zip manually at the end
    user_agent="colabfold/google-colab-main",
)

# Package results into a zip file
print("Packaging results...")
results_zip_path = f"{jobname}.result.zip"
os.system(f"zip -r -q {results_zip_path} {jobname}")

print(f"\nDone! Prediction complete.")
print(f"Results are saved in '{results_zip_path}'.")

In [ ]:
#@title ▶️ Run ALyzer3D.AI Analysis
import os
import sys
import glob
import json
import numpy as np
import joblib
import torch
import tensorflow as tf
from IPython.display import display, HTML

# ------------------------------------------------------------------------------
# 1. INSTALLATION & SETUP
# ------------------------------------------------------------------------------
print("✅ Step 1: Installing ALyzer3D.AI dependencies...")
!pip install -q transformers scikit-learn joblib biopython > /dev/null 2>&1

print("   Cloning repository...")
if not os.path.exists('/content/ALyzer3D.AI'):
    !git clone https://github.com/petercmay89/ALyzer3D.AI.git > /dev/null 2>&1
sys.path.insert(0, '/content/ALyzer3D.AI')

from transformers import AutoTokenizer, EsmModel
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.PDBData import protein_letters_3to1
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# ------------------------------------------------------------------------------
# 2. PREDICTOR CLASS DEFINITION (Based on new script)
# ------------------------------------------------------------------------------
print("\n✅ Step 2: Initializing the Amyloid Predictor...")

# Configuration
MODEL_FOLDER_NAME = "paper_model_scalar_pathway_v1_minus5_stripped_80_20_seed3"
REPO_PATH = "/content/ALyzer3D.AI"
FULL_MODEL_DIR = os.path.join(REPO_PATH, MODEL_FOLDER_NAME)
MAX_LENGTH = 120
PLM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

class AmyloidPredictor:
    def __init__(self, model_dir):
        self.model_dir = model_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"   -> Loading ESM-2 Model ({PLM_MODEL_NAME})...")
        self.tokenizer = AutoTokenizer.from_pretrained(PLM_MODEL_NAME)
        self.plm_model = EsmModel.from_pretrained(PLM_MODEL_NAME).to(self.device)
        self.plm_model.eval()

        print(f"   -> Loading Ensemble models from: {os.path.basename(model_dir)}")
        self.models = []
        self.scalers = []
        self._load_ensemble()

    def _load_ensemble(self):
        if not os.path.exists(self.model_dir):
            raise FileNotFoundError(f"Directory '{self.model_dir}' does not exist.")

        model_files = sorted(glob.glob(os.path.join(self.model_dir, "*.keras")))
        if not model_files:
            model_files = sorted(glob.glob(os.path.join(self.model_dir, "*.h5")))
        scaler_files = sorted(glob.glob(os.path.join(self.model_dir, "*.joblib")))

        if not model_files or len(model_files) != len(scaler_files):
            raise ValueError(f"Found {len(model_files)} models and {len(scaler_files)} scalers. Mismatch or empty.")

        for mp, sp in zip(model_files, scaler_files):
            # safe_mode=False is required for Lambda layers
            self.models.append(tf.keras.models.load_model(mp, compile=False, safe_mode=False))
            self.scalers.append(joblib.load(sp))

    def _get_embedding(self, sequence):
        inputs = self.tokenizer(
            sequence, return_tensors="pt", truncation=True, max_length=1022
        ).to(self.device)
        with torch.no_grad():
            outputs = self.plm_model(**inputs)
        return outputs.last_hidden_state.squeeze(0).mean(dim=0).cpu().numpy()

    def _calculate_rog(self, pdb_path):
        try:
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure("s", pdb_path)
            model = structure[0]
            atoms = list(model.get_atoms())
            if not atoms: return 0.0
            com = sum(a.coord for a in atoms) / len(atoms)
            rog_sq = sum(np.sum((a.coord - com)**2) for a in atoms)
            n_res = len(list(model.get_residues()))
            return np.sqrt(rog_sq / len(atoms)) / np.sqrt(n_res) if n_res > 0 else 0.0
        except:
            return 0.0

    def _get_biochem_features(self, sequence):
        try:
            seq = "".join(c for c in sequence if c in "ACDEFGHIKLMNPQRSTVWY")
            pa = ProteinAnalysis(seq)
            return [pa.isoelectric_point(), pa.gravy(), pa.aromaticity(), pa.molecular_weight()]
        except:
            return [7.0, 0.0, 0.0, 12000.0]

    def _load_sequence(self, pdb_path):
        parser = PDBParser(QUIET=True)
        chain = parser.get_structure("s", pdb_path)[0].get_chains().__next__()
        return "".join(
            protein_letters_3to1.get(r.get_resname().upper(), 'X')
            for r in chain.get_residues() if is_aa(r, standard=True)
        )

    def predict(self, pdb_path, json_path):
        try:
            sequence = self._load_sequence(pdb_path)
            with open(json_path, 'r') as f:
                data = json.load(f)
        except Exception as e:
            return {"error": f"File loading failed: {e}"}

        plddt = np.array(data['plddt'])
        pae = np.array(data['pae'])

        L_struct = min(len(sequence), len(plddt), pae.shape[0])
        effective_len = L_struct - 5

        if effective_len <= 0:
            return {"error": f"Protein too short (len={L_struct}) for -5 truncation."}

        slice_len = min(effective_len, MAX_LENGTH)

        pad_pae = np.zeros((MAX_LENGTH, MAX_LENGTH))
        pad_pae[:slice_len, :slice_len] = pae[:slice_len, :slice_len]

        pad_plddt = np.zeros(MAX_LENGTH)
        pad_plddt[:slice_len] = plddt[:slice_len]

        pad_row = np.zeros(MAX_LENGTH)
        pad_col = np.zeros(MAX_LENGTH)
        if slice_len > 0:
            pad_row[:slice_len] = np.mean(pae[:slice_len, :slice_len], axis=1)
            pad_col[:slice_len] = np.mean(pae[:slice_len, :slice_len], axis=0)

        embedding = self._get_embedding(sequence)
        biochem = self._get_biochem_features(sequence)
        rog = self._calculate_rog(pdb_path)
        raw_scalars = np.array(biochem + [rog]).reshape(1, -1)

        inputs_base = {
            "pae_input": np.expand_dims(pad_pae, [0, -1]),
            "plddt_input": np.expand_dims(pad_plddt, [0, -1]),
            "embedding_input": np.expand_dims(embedding, 0),
            "pae_row_input": np.expand_dims(pad_row, [0, -1]),
            "pae_col_input": np.expand_dims(pad_col, [0, -1]),
            "length_input": np.array([effective_len])
        }

        fold_preds = []
        for model, scaler in zip(self.models, self.scalers):
            inputs_fold = inputs_base.copy()
            inputs_fold["scalar_features_input"] = scaler.transform(raw_scalars)
            pred = model.predict(inputs_fold, verbose=0)[0][0]
            fold_preds.append(pred)

        avg_prob = np.mean(fold_preds)

        return {
            "sequence": sequence,
            "label": "AMYLOID" if avg_prob > 0.5 else "NON-AMYLOID",
            "probability": float(avg_prob),
            "fold_scores": fold_preds
        }

# Initialize the predictor
try:
    predictor = AmyloidPredictor(model_dir=FULL_MODEL_DIR)
    print(" ✔️ Predictor loaded successfully.")
except Exception as e:
    print(f"❗️ Error loading model: {e}")

# ------------------------------------------------------------------------------
# 3. RUN PREDICTION
# ------------------------------------------------------------------------------
print("\n✅ Step 3: Running analysis on ColabFold Output...")

# Locate files from the jobname in the previous cell
search_path_pdb = f"{jobname}/{jobname}_unrelaxed_rank_001*.pdb"
search_path_json = f"{jobname}/{jobname}_scores_rank_001*.json"

pdb_files = glob.glob(search_path_pdb)
json_files = glob.glob(search_path_json)

if not pdb_files or not json_files:
    print(f"❗️ Error: Could not find output files inside the '{jobname}' folder. Please check the file browser.")
else:
    pdb_filename = pdb_files[0]
    json_filename = json_files[0]
    print(f" - Found PDB: {pdb_filename}")
    print(f" - Found JSON: {json_filename}")

    # Run the prediction
    result = predictor.predict(pdb_path=pdb_filename, json_path=json_filename)

    # --------------------------------------------------------------------------
    # 4. SAVE RESULTS TO CSV & DISPLAY
    # --------------------------------------------------------------------------
    if result.get("error"):
        print(f"❗️ An error occurred during analysis: {result['error']}")
    else:
        prob = result['probability']
        label = result['label']
        confidence_percent = prob * 100

        # --- NEW SECTION: SAVE CSV ---
        print("   -> Saving analysis results to CSV...")

        # Create a dictionary for the results
        csv_data = [{
            "ID": jobname,
            "Prediction": label,
            "Probability": prob,
            "Confidence_Percent": round(confidence_percent, 2),
            "Sequence": result['sequence']
        }]

        # Save to a new CSV file inside the job folder
        results_csv_name = f"{jobname}_alyzer_results.csv"
        results_csv_path = os.path.join(jobname, results_csv_name)

        import pandas as pd

        df = pd.DataFrame(csv_data)
        df.to_csv(results_csv_path, index=False)
        print(f"   -> CSV created: {results_csv_path}")

        # UPDATE THE ZIP FILE (Add the new CSV to the existing zip)
        print("   -> Updating Zip file...")
        zip_file_name = f"{jobname}.result.zip"
        if os.path.exists(zip_file_name):
            # The 'zip -u' command updates the zip file with the new file
            os.system(f"zip -u {zip_file_name} {results_csv_path}")
            print(f"   -> {zip_file_name} updated successfully.")
        else:
            print("   ⚠️ Warning: Zip file not found to update.")

        # --- END NEW SECTION ---

    # --------------------------------------------------------------------------
    # 4. DISPLAY RESULTS
    # --------------------------------------------------------------------------
    if result.get("error"):
        print(f"❗️ An error occurred during analysis: {result['error']}")
    else:
        prob = result['probability']
        label = result['label']
        confidence_percent = prob * 100

        # Determine Risk Colors based on the result
        # Note: Assuming High Risk = Amyloid (High Prob)
        if prob > 0.6:
            risk_level = "High Risk (Amyloid)"
            risk_color = "#D32F2F" # Red
        else:
            risk_level = "Low Risk (Non-Amyloid)"
            risk_color = "#388E3C" # Green

        html_output = f"""
        <div style="border: 2px solid {risk_color}; border-radius: 10px; padding: 20px; font-family: sans-serif; background-color: #f9f9f9; margin-top: 1em;">
            <h2 style="color: {risk_color}; margin-top: 0;">ANALYSIS COMPLETE: {risk_level.upper()}</h2>
            <hr>
            <div style="display: grid; grid-template-columns: 150px 1fr; gap: 10px; align-items: center;">

                <strong style="font-size: 1.1em;">Prediction:</strong>
                <span style="font-size: 1.1em; font-weight: bold; color: {risk_color};">{label}</span>

                <strong style="font-size: 1.1em;">Probability:</strong>
                <div style="width: 100%; background-color: #e0e0e0; border-radius: 5px;">
                    <div style="width: {confidence_percent}%; background-color: {risk_color}; color: white; text-align: center; padding: 2px 0; border-radius: 5px; min-width: 30px;">
                        {confidence_percent:.2f}%
                    </div>
                </div>

                <strong style="vertical-align: top;">Sequence:</strong>
                <textarea readonly style="width: 100%; height: 60px; resize: none; border: 1px solid #ccc; font-family: monospace; background-color: #fff;">{result['sequence']}</textarea>
            </div>
        </div>
        """
        display(HTML(html_output))

In [ ]:
#@title Download Results
from google.colab import files

# Download the zip file created in the prediction cell
files.download(f"{jobname}.result.zip")

# Instructions <a name="Instructions"></a>
For detailed instructions, tips and tricks on ColabFold, see recently published paper at [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5)